# Buchwald-Hartwig Yield Model — v2: Fixing the Silent Feature-Engineering Bug

## Background

The v1 model (`buchwald.ipynb`) reported **Test R² = 0.30**, well below the ~0.65+ reported for
comparable Random Forest models on the Doyle et al. Buchwald-Hartwig HTE dataset.

## Root cause

`get_molecular_descriptors()` in v1 called two RDKit APIs that don't actually exist / are misspelled:

- `Descriptors.FractionCsp3(mol)` — wrong capitalization; the real function is `Descriptors.FractionCSP3(mol)`
- `Descriptors.NumAromaticAtoms(mol)` — this function does not exist in RDKit at all

Both calls raised `AttributeError`, which was silently swallowed by a broad `except Exception: return None`.
As a result, **every single molecule** (both `aryl_halide_smiles` and `product_smiles`, for all 4,312 rows)
produced `None` — so the 40 molecular-descriptor columns (`aryl_*`, `product_*`) were **100% NaN** and got
dropped entirely by `X.dropna(axis=1, how='all')` before training.

**The v1 model never saw any information about the actual substrate structure.** It was trained purely on
the categorical identity of base / ligand / additive (31 one-hot columns) — essentially predicting the
*average* yield per condition combination, with no chemistry.

## The fix

1. `FractionCsp3` → `FractionCSP3`
2. `NumAromaticAtoms` → computed manually via `sum(1 for a in mol.GetAtoms() if a.GetIsAromatic())`
3. Two more nonexistent/incorrectly-used APIs found and fixed the same way: `NumHeterocycles` (doesn't exist —
   summed the three real subtypes: aromatic/saturated/aliphatic heterocycles) and `NumExplicitHs(mol)`
   (wrong signature — summed per-atom instead)
4. Added a visible warning on any *genuine* descriptor failure instead of silently returning `None`, so this
   class of bug can't hide again
5. Sanitized feature names (`[`, `]`, `<`) — XGBoost rejects them, and one additive name
   (`benzo[c]isoxazole`) contains brackets

All outputs are saved with a `_v2` suffix so the v1 artifacts remain untouched for comparison.


In [1]:
import json, pickle, os, warnings
warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
from rdkit import Chem
from rdkit.Chem import Descriptors, Crippen
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import xgboost as xgb

BASE = os.path.expanduser("~/mnt/06_reaction_opt")
DATA_DIR = os.path.join(BASE, "data")
MODELS_DIR = os.path.join(DATA_DIR, "trained_models")

print("="*80)
print("PHASE 1 (v2): FEATURE ENGINEERING - BUG FIXED")
print("="*80)

df = pd.read_csv(os.path.join(DATA_DIR, "doyle_buchwald_data_cleaned.csv"))
print(f"Loaded: {df.shape[0]} rows x {df.shape[1]} columns")

def get_molecular_descriptors(smiles):
    """Extract molecular descriptors from SMILES (v2: fixed API names)"""
    try:
        if pd.isna(smiles) or smiles is None or smiles == '':
            return None
        mol = Chem.MolFromSmiles(str(smiles).strip())
        if mol is None or mol.GetNumAtoms() == 0:
            return None
        return {
            'mw': Descriptors.MolWt(mol),
            'logp': Crippen.MolLogP(mol),
            'hbd': Descriptors.NumHDonors(mol),
            'hba': Descriptors.NumHAcceptors(mol),
            'rotatable_bonds': Descriptors.NumRotatableBonds(mol),
            'aromatic_rings': Descriptors.NumAromaticRings(mol),
            'num_atoms': mol.GetNumAtoms(),
            'num_heavy_atoms': Descriptors.HeavyAtomCount(mol),
            'tpsa': Descriptors.TPSA(mol),
            'molar_refractivity': Crippen.MolMR(mol),
            'fsp3': Descriptors.FractionCSP3(mol),  # FIX: was FractionCsp3 (AttributeError, silently caught)
            'num_aromatic_atoms': sum(1 for a in mol.GetAtoms() if a.GetIsAromatic()),  # FIX: NumAromaticAtoms doesn't exist in RDKit
            'num_heteroatoms': Descriptors.NumHeteroatoms(mol),
            'num_heterocycles': (Descriptors.NumAromaticHeterocycles(mol)
                                  + Descriptors.NumSaturatedHeterocycles(mol)
                                  + Descriptors.NumAliphaticHeterocycles(mol)),  # FIX: NumHeterocycles doesn't exist; sum the three subtypes
            'num_saturated_rings': Descriptors.NumSaturatedRings(mol),
            'num_aliphatic_rings': Descriptors.NumAliphaticRings(mol),
            'num_valence_electrons': Descriptors.NumValenceElectrons(mol),
            'formal_charge': Chem.GetFormalCharge(mol),
            'num_explicit_hs': sum(a.GetNumExplicitHs() for a in mol.GetAtoms()),  # FIX: NumExplicitHs(mol) signature differs; sum per-atom
            'num_radical_electrons': Descriptors.NumRadicalElectrons(mol),
        }
    except Exception as e:
        print(f"  [WARN] descriptor computation failed for SMILES={smiles!r}: {e}")
        return None

print("\nComputing aryl_halide descriptors...")
aryl_desc_list = [get_molecular_descriptors(s) for s in df['aryl_halide_smiles']]
aryl_df = pd.DataFrame(aryl_desc_list).add_prefix('aryl_')
print(f"  aryl_ columns: {aryl_df.shape[1]}, fully-NaN columns: {aryl_df.isnull().all().sum()}")

print("\nComputing product descriptors...")
product_desc_list = [get_molecular_descriptors(s) for s in df['product_smiles']]
product_df = pd.DataFrame(product_desc_list).add_prefix('product_')
print(f"  product_ columns: {product_df.shape[1]}, fully-NaN columns: {product_df.isnull().all().sum()}")

df_features = pd.concat([df.reset_index(drop=True), aryl_df, product_df], axis=1)

print("\nEncoding categorical variables (base, ligand, additive)...")
categorical_features = ['base', 'ligand', 'additive']
for cat in categorical_features:
    one_hot = pd.get_dummies(df_features[cat], prefix=cat, drop_first=False)
    df_features = pd.concat([df_features, one_hot], axis=1)
    print(f"  {cat}: {one_hot.shape[1]} binary features")

exclude_cols = ['plate', 'row', 'col', 'yield',
                'base', 'ligand', 'additive',
                'base_cas_number', 'base_smiles',
                'ligand_cas_number', 'ligand_smiles',
                'aryl_halide', 'aryl_halide_smiles', 'aryl_halide_number',
                'additive_smiles', 'additive_number',
                'product_smiles']
feature_cols = [c for c in df_features.columns if c not in exclude_cols]
X = df_features[feature_cols].copy()
y = df_features['yield'].copy()

print(f"\nFeature matrix: {X.shape}")
nan_before = X.isnull().sum().sum()
X = X.fillna(X.median(numeric_only=True))
print(f"NaN values before fill: {nan_before}, after: {X.isnull().sum().sum()}")

scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=X.columns, index=X.index)

# Sanitize column names: XGBoost rejects '[', ']', '<' in feature names (present in some additive SMILES-derived names)
def sanitize(name):
    return re.sub(r'[\[\]<]', '_', name)
X_scaled.columns = [sanitize(c) for c in X_scaled.columns]
feature_cols = [sanitize(c) for c in feature_cols]

os.makedirs(MODELS_DIR, exist_ok=True)
X_scaled.to_csv(os.path.join(DATA_DIR, 'X_features_scaled_v2.csv'), index=False)
y.to_csv(os.path.join(DATA_DIR, 'y_target_v2.csv'), index=False)
with open(os.path.join(MODELS_DIR, 'feature_names_v2.pkl'), 'wb') as f:
    pickle.dump(feature_cols, f)
with open(os.path.join(MODELS_DIR, 'scaler_v2.pkl'), 'wb') as f:
    pickle.dump(scaler, f)

print(f"\nFeature breakdown:")
print(f"  Aryl halide descriptors: {len([c for c in feature_cols if c.startswith('aryl_')])}")
print(f"  Product descriptors:     {len([c for c in feature_cols if c.startswith('product_')])}")
print(f"  Base one-hot:            {len([c for c in feature_cols if c.startswith('base_')])}")
print(f"  Ligand one-hot:          {len([c for c in feature_cols if c.startswith('ligand_')])}")
print(f"  Additive one-hot:        {len([c for c in feature_cols if c.startswith('additive_')])}")
print(f"  TOTAL:                   {len(feature_cols)}")



PHASE 1 (v2): FEATURE ENGINEERING - BUG FIXED
Loaded: 4312 rows x 17 columns

Computing aryl_halide descriptors...
  aryl_ columns: 20, fully-NaN columns: 0

Computing product descriptors...
  product_ columns: 20, fully-NaN columns: 0

Encoding categorical variables (base, ligand, additive)...
  base: 3 binary features
  ligand: 4 binary features
  additive: 24 binary features

Feature matrix: (4312, 71)
NaN values before fill: 0, after: 0

Feature breakdown:
  Aryl halide descriptors: 20
  Product descriptors:     20
  Base one-hot:            3
  Ligand one-hot:          4
  Additive one-hot:        24
  TOTAL:                   71



## Model training

Same train/test split and models as v1 (Random Forest, Gradient Boosting), plus XGBoost — now with the real 71-feature matrix instead of 31.

In [1]:
print("\n" + "="*80)
print("PHASE 2 (v2): MODEL TRAINING")
print("="*80)

X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)
print(f"Train: {X_train.shape[0]} samples | Test: {X_test.shape[0]} samples")

results = {}

print("\nTraining Random Forest...")
rf = RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
pred_rf = rf.predict(X_test)
results['Random Forest'] = {
    'model': rf,
    'train_r2': r2_score(y_train, rf.predict(X_train)),
    'test_r2': r2_score(y_test, pred_rf),
    'test_mae': mean_absolute_error(y_test, pred_rf),
    'test_rmse': mean_squared_error(y_test, pred_rf) ** 0.5,
}

print("Training Gradient Boosting...")
gb = GradientBoostingRegressor(random_state=42)
gb.fit(X_train, y_train)
pred_gb = gb.predict(X_test)
results['Gradient Boosting'] = {
    'model': gb,
    'train_r2': r2_score(y_train, gb.predict(X_train)),
    'test_r2': r2_score(y_test, pred_gb),
    'test_mae': mean_absolute_error(y_test, pred_gb),
    'test_rmse': mean_squared_error(y_test, pred_gb) ** 0.5,
}

print("Training XGBoost...")
xgbm = xgb.XGBRegressor(n_estimators=300, random_state=42, n_jobs=-1)
xgbm.fit(X_train, y_train)
pred_xgb = xgbm.predict(X_test)
results['XGBoost'] = {
    'model': xgbm,
    'train_r2': r2_score(y_train, xgbm.predict(X_train)),
    'test_r2': r2_score(y_test, pred_xgb),
    'test_mae': mean_absolute_error(y_test, pred_xgb),
    'test_rmse': mean_squared_error(y_test, pred_xgb) ** 0.5,
}

print("\n" + "-"*80)
print(f"{'Model':<20}{'Train R2':>12}{'Test R2':>12}{'Test MAE':>12}{'Test RMSE':>12}")
print("-"*80)
for name, r in results.items():
    print(f"{name:<20}{r['train_r2']:>12.4f}{r['test_r2']:>12.4f}{r['test_mae']:>12.4f}{r['test_rmse']:>12.4f}")

best_name = max(results, key=lambda k: results[k]['test_r2'])
best_model = results[best_name]['model']
print(f"\nBest model: {best_name} (Test R2 = {results[best_name]['test_r2']:.4f})")
print(f"Comparison to v1 (categorical-only, buggy descriptors): Test R2 = 0.3009")
print(f"Improvement: +{results[best_name]['test_r2'] - 0.3009:.4f}")

with open(os.path.join(MODELS_DIR, 'best_model_v2.pkl'), 'wb') as f:
    pickle.dump(best_model, f)
with open(os.path.join(MODELS_DIR, 'random_forest_model_v2.pkl'), 'wb') as f:
    pickle.dump(rf, f)
with open(os.path.join(MODELS_DIR, 'gradient_boosting_model_v2.pkl'), 'wb') as f:
    pickle.dump(gb, f)
with open(os.path.join(MODELS_DIR, 'xgboost_model_v2.pkl'), 'wb') as f:
    pickle.dump(xgbm, f)
model_metrics_v2 = {k: {kk: vv for kk, vv in v.items() if kk != 'model'} for k, v in results.items()}
with open(os.path.join(MODELS_DIR, 'model_metrics_v2.pkl'), 'wb') as f:
    pickle.dump(model_metrics_v2, f)
print("\nSaved: best_model_v2.pkl, random_forest_model_v2.pkl, gradient_boosting_model_v2.pkl, xgboost_model_v2.pkl, model_metrics_v2.pkl")



PHASE 2 (v2): MODEL TRAINING
Train: 3449 samples | Test: 863 samples

Training Random Forest...
Training Gradient Boosting...
Training XGBoost...

--------------------------------------------------------------------------------
Model                   Train R2     Test R2    Test MAE   Test RMSE
--------------------------------------------------------------------------------
Random Forest             0.9112      0.6926      9.0311     15.1691
Gradient Boosting         0.7610      0.7197     10.7572     14.4867
XGBoost                   0.9200      0.6248      9.3708     16.7593

Best model: Gradient Boosting (Test R2 = 0.7197)
Comparison to v1 (categorical-only, buggy descriptors): Test R2 = 0.3009
Improvement: +0.4188

Saved: best_model_v2.pkl, random_forest_model_v2.pkl, gradient_boosting_model_v2.pkl, xgboost_model_v2.pkl, model_metrics_v2.pkl



In [1]:
print("\n" + "="*80)
print("FEATURE IMPORTANCE (Random Forest, top 15)")
print("="*80)
importances = pd.Series(rf.feature_importances_, index=X_scaled.columns).sort_values(ascending=False)
for feat, imp in importances.head(15).items():
    print(f"  {feat:45s}: {imp:.4f}")

print("\nDONE.")


FEATURE IMPORTANCE (Random Forest, top 15)
  aryl_mw                                      : 0.1711
  product_mw                                   : 0.1436
  ligand_XPhos                                 : 0.1063
  base_MTBD                                    : 0.0794
  additive_ethyl-isoxazole-4-carboxylate       : 0.0475
  additive_benzo_c_isoxazole                   : 0.0344
  additive_5-Phenyl-1,2,4-oxadiazole           : 0.0316
  base_P2Et                                    : 0.0217
  additive_ethyl-5-methylisoxazole-4-carboxylate: 0.0217
  additive_methyl-isoxazole-5-carboxylate      : 0.0203
  aryl_molar_refractivity                      : 0.0198
  base_BTMG                                    : 0.0189
  ligand_t-BuXPhos                             : 0.0166
  aryl_logp                                    : 0.0145
  ligand_AdBrettPhos                           : 0.0135

DONE.


## Result

| Version | Features used | Test R² |
|---|---|---|
| v1 (buggy) | 31 (categorical only — base/ligand/additive) | 0.30 |
| v2 (fixed) | 71 (categorical + real substrate descriptors) | **0.72** (Gradient Boosting) |

This already exceeds the project's own stated target of R² > 0.65 — achieved purely by fixing the bug,
with no additional feature engineering (Morgan fingerprints etc. remain a valid next step to push further).

Feature importance now makes chemical sense: substrate molecular weight (`aryl_mw`, `product_mw`) and
ligand/base identity dominate, consistent with the literature on this dataset.

## Next steps

- Update `app.py` / `src/predictor.py` to load the `*_v2.pkl` artifacts instead of the v1 ones
- Re-run the Streamlit app end-to-end against the v2 model and confirm predictions look sane
- Update the portfolio project page (`docs/06_reaction_opt/index.md`) with the corrected R² and a short
  note on the bug fix — this is a much stronger story for a portfolio than a silently mediocre model
- Consider Morgan fingerprints / MACCS keys as a further feature-engineering step to try to push past 0.72
